
# 📈 Backtester de rebotes — RSI + Bollinger + Volumen

Este notebook implementa una lógica de **proceso**, no de señales aisladas:

**FUERA → CAÍDA DETECTADA → VIGILANCIA → REBOTE CONFIRMADO → COMPRA → POSIBLE TECHO → GIRO CONFIRMADO → VENTA**

Objetivo: evitar comprar mientras el precio todavía está cayendo con fuerza.

Características:

- Descarga histórico con `yfinance`
- Ticker configurable
- Rango gráfico configurable
- Ventana operativa configurable
- Capital inicial configurable
- Interés compuesto
- Detección de caída fuerte con RSI + Bollinger
- Confirmación de rebote con precio + RSI + volumen
- Detección de techo con RSI + Bollinger
- Confirmación de giro bajista con precio + RSI + volumen
- Stop por estructura: debajo del mínimo reciente del suelo
- Tabla de operaciones
- Curva de capital
- Si queda una operación abierta al final de la ventana, se anula


In [ ]:

# ================================================================
# ⚙️ CELDA 1 — CONFIGURACIÓN PRINCIPAL
# ================================================================
# 👉 MODIFICA NORMALMENTE SOLO ESTA CELDA
# ================================================================

# ------------------------------------------------
# 1️⃣ VALOR
# ------------------------------------------------
# DEJA ACTIVA SOLO UNA LÍNEA

TICKER = "META"        # Meta
# TICKER = "AMZN"      # Amazon
# TICKER = "NVDA"      # NVIDIA
# TICKER = "MSFT"      # Microsoft
# TICKER = "AAPL"      # Apple
# TICKER = "GOOGL"     # Alphabet
# TICKER = "AVGO"      # Broadcom
# TICKER = "TSLA"      # Tesla
# TICKER = "COST"      # Costco
# TICKER = "NFLX"      # Netflix


# ------------------------------------------------
# 2️⃣ RANGO TOTAL DEL GRÁFICO / DESCARGA
# ------------------------------------------------
# Formato AAAA-MM-DD
# Conviene descargar margen ANTERIOR a la ventana operativa
# para que RSI/Bollinger/volumen estén bien calculados.

FECHA_GRAFICA_INICIO = "2025-08-20"
FECHA_GRAFICA_FIN    = "2026-08-20"


# ------------------------------------------------
# 3️⃣ VENTANA OPERATIVA
# ------------------------------------------------
# El algoritmo SOLO puede abrir operaciones dentro de esta ventana.

FECHA_OPERATIVA_INICIO = "2026-01-01"
FECHA_OPERATIVA_FIN    = "2026-06-01"


# ------------------------------------------------
# 4️⃣ CAPITAL
# ------------------------------------------------

CAPITAL_INICIAL = 10000.0
REINVERTIR_TODO = True


# ------------------------------------------------
# 5️⃣ RSI
# ------------------------------------------------

RSI_PERIODO = 14

# Zona de caída/suelo
RSI_SUELO = 35

# Zona de techo
RSI_TECHO = 68


# ------------------------------------------------
# 6️⃣ BOLLINGER
# ------------------------------------------------

BB_PERIODO = 20
BB_DESVIACIONES = 2.0


# ------------------------------------------------
# 7️⃣ VOLUMEN
# ------------------------------------------------

VOLUMEN_PERIODO = 20

# Confirmación de rebote:
# volumen actual > media * multiplicador
VOLUMEN_REBOTE_MULT = 1.10

# Confirmación de giro bajista:
VOLUMEN_SALIDA_MULT = 1.05


# ------------------------------------------------
# 8️⃣ DETECCIÓN DE CAÍDA FUERTE
# ------------------------------------------------

# Caída mínima acumulada en N sesiones para activar vigilancia.
CAIDA_SESIONES = 5
CAIDA_MIN_PCT = -6.0

# Cuántas sesiones esperamos un rebote tras detectar posible suelo.
DIAS_VIGILANCIA_SUELO = 7


# ------------------------------------------------
# 9️⃣ CONFIRMACIÓN DEL REBOTE
# ------------------------------------------------
# La compra exige estas condiciones.
#
# - RSI subiendo
# - cierre > cierre anterior
# - cierre vuelve dentro de Bollinger
# - cierre > máximo de la vela anterior
# - volumen relevante

EXIGIR_ROMPER_MAXIMO_ANTERIOR = True
EXIGIR_VOLUMEN_REBOTE = True

# Podemos exigir que el RSI haya recuperado un mínimo antes de comprar.
RSI_MIN_REBOTE = 28


# ------------------------------------------------
# 🔟 DETECCIÓN DE POSIBLE TECHO
# ------------------------------------------------

# Subida acumulada mínima reciente para considerar sobreextensión.
SUBIDA_SESIONES = 5
SUBIDA_MIN_PCT = 5.0

# Cuántos días esperamos confirmación bajista.
DIAS_VIGILANCIA_TECHO = 7


# ------------------------------------------------
# 1️⃣1️⃣ CONFIRMACIÓN DE SALIDA
# ------------------------------------------------
# - RSI bajando
# - cierre < cierre anterior
# - vuelve dentro desde la banda superior
# - cierre < mínimo de vela anterior
# - volumen relevante

EXIGIR_ROMPER_MINIMO_ANTERIOR = True
EXIGIR_VOLUMEN_SALIDA = True


# ------------------------------------------------
# 1️⃣2️⃣ STOP POR ESTRUCTURA
# ------------------------------------------------
# Se coloca debajo del mínimo observado durante la fase de suelo.

STOP_MARGEN_PCT = 1.5

# Si el precio avanza a favor, podemos subir el stop
# bajo el mínimo de las últimas N sesiones.
USAR_TRAILING_ESTRUCTURAL = True
TRAILING_MIN_SESIONES = 3
TRAILING_MARGEN_PCT = 1.0


# ------------------------------------------------
# 1️⃣3️⃣ COSTES
# ------------------------------------------------
# 0.10 = 0,10 %

COMISION_PCT = 0.0
SLIPPAGE_PCT = 0.0


# ------------------------------------------------
# 1️⃣4️⃣ FILTROS EXTRA
# ------------------------------------------------

# Evita volver a entrar inmediatamente después de una salida.
DIAS_ESPERA_TRAS_SALIDA = 2


print("=" * 72)
print("📊 CONFIGURACIÓN")
print("=" * 72)
print(f"Ticker:                 {TICKER}")
print(f"Gráfico:                {FECHA_GRAFICA_INICIO} → {FECHA_GRAFICA_FIN}")
print(f"Ventana operativa:      {FECHA_OPERATIVA_INICIO} → {FECHA_OPERATIVA_FIN}")
print(f"Capital inicial:        ${CAPITAL_INICIAL:,.2f}")
print(f"RSI suelo / techo:      {RSI_SUELO} / {RSI_TECHO}")
print(f"Caída mínima:           {CAIDA_MIN_PCT}% en {CAIDA_SESIONES} sesiones")
print(f"Volumen rebote:         {VOLUMEN_REBOTE_MULT:.2f}x media")
print(f"Stop bajo mínimo suelo: {STOP_MARGEN_PCT:.2f}%")
print("=" * 72)


In [ ]:

# ================================================================
# 📦 CELDA 2 — LIBRERÍAS
# ================================================================

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

print("✅ Librerías cargadas")


In [ ]:

# ================================================================
# 📥 CELDA 3 — DESCARGAR HISTÓRICO
# ================================================================

print(f"📥 Descargando {TICKER}...")

df = yf.download(
    TICKER,
    start=FECHA_GRAFICA_INICIO,
    end=FECHA_GRAFICA_FIN,
    interval="1d",
    auto_adjust=False,
    progress=False
)

if df.empty:
    raise ValueError("No se han descargado datos. Revisa ticker y fechas.")

df = df.reset_index()

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

columnas = ["Date", "Open", "High", "Low", "Close", "Volume"]
df = df[[x for x in columnas if x in df.columns]].copy()

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").drop_duplicates("Date").reset_index(drop=True)

print(f"✅ {len(df)} sesiones descargadas")
print(f"Desde: {df['Date'].min().date()}")
print(f"Hasta: {df['Date'].max().date()}")

display(df.tail())


In [ ]:

# ================================================================
# 🧮 CELDA 4 — INDICADORES
# ================================================================

data = df.copy()

# ------------------------------------------------
# RSI (Wilder)
# ------------------------------------------------

delta = data["Close"].diff()

ganancia = delta.clip(lower=0)
perdida = -delta.clip(upper=0)

avg_gain = ganancia.ewm(
    alpha=1 / RSI_PERIODO,
    adjust=False,
    min_periods=RSI_PERIODO
).mean()

avg_loss = perdida.ewm(
    alpha=1 / RSI_PERIODO,
    adjust=False,
    min_periods=RSI_PERIODO
).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)

data["RSI"] = 100 - (100 / (1 + rs))


# ------------------------------------------------
# BOLLINGER
# ------------------------------------------------

data["BB_Media"] = data["Close"].rolling(BB_PERIODO).mean()

bb_std = data["Close"].rolling(BB_PERIODO).std(ddof=0)

data["BB_Sup"] = data["BB_Media"] + BB_DESVIACIONES * bb_std
data["BB_Inf"] = data["BB_Media"] - BB_DESVIACIONES * bb_std


# ------------------------------------------------
# VOLUMEN
# ------------------------------------------------

data["Vol_Media"] = data["Volume"].rolling(VOLUMEN_PERIODO).mean()


# ------------------------------------------------
# VARIACIONES ACUMULADAS
# ------------------------------------------------

data["Caida_pct"] = (
    data["Close"] / data["Close"].shift(CAIDA_SESIONES) - 1
) * 100

data["Subida_pct"] = (
    data["Close"] / data["Close"].shift(SUBIDA_SESIONES) - 1
) * 100


print("✅ Indicadores calculados")

display(
    data[
        [
            "Date", "Close", "RSI",
            "BB_Inf", "BB_Media", "BB_Sup",
            "Volume", "Vol_Media",
            "Caida_pct", "Subida_pct"
        ]
    ].tail(10)
)


In [ ]:

# ================================================================
# 🚦 CELDA 5 — SEÑALES DEL PROCESO
# ================================================================

sig = data.copy()

# ------------------------------------------------
# A) DETECTAR CAÍDA FUERTE / POSIBLE SUELO
# ------------------------------------------------
#
# Se activa vigilancia cuando:
# - RSI está en zona baja
# - precio toca/perfora banda inferior
# - y existe caída acumulada relevante

sig["Caida_Fuerte"] = (
    (sig["RSI"] <= RSI_SUELO) &
    (sig["Close"] <= sig["BB_Inf"]) &
    (sig["Caida_pct"] <= CAIDA_MIN_PCT)
)


# ------------------------------------------------
# B) CONFIRMAR REBOTE
# ------------------------------------------------

sig["RSI_Subiendo"] = sig["RSI"] > sig["RSI"].shift(1)
sig["Precio_Subiendo"] = sig["Close"] > sig["Close"].shift(1)
sig["Dentro_Bollinger_Inf"] = sig["Close"] > sig["BB_Inf"]

sig["Rompe_Max_Anterior"] = (
    sig["Close"] > sig["High"].shift(1)
)

sig["Volumen_Rebote_OK"] = (
    sig["Volume"] >= sig["Vol_Media"] * VOLUMEN_REBOTE_MULT
)

cond_rebote = (
    sig["RSI_Subiendo"] &
    sig["Precio_Subiendo"] &
    sig["Dentro_Bollinger_Inf"] &
    (sig["RSI"] >= RSI_MIN_REBOTE)
)

if EXIGIR_ROMPER_MAXIMO_ANTERIOR:
    cond_rebote &= sig["Rompe_Max_Anterior"]

if EXIGIR_VOLUMEN_REBOTE:
    cond_rebote &= sig["Volumen_Rebote_OK"]

sig["Rebote_Confirmado"] = cond_rebote


# ------------------------------------------------
# C) DETECTAR POSIBLE TECHO
# ------------------------------------------------

sig["Posible_Techo"] = (
    (sig["RSI"] >= RSI_TECHO) &
    (sig["Close"] >= sig["BB_Sup"]) &
    (sig["Subida_pct"] >= SUBIDA_MIN_PCT)
)


# ------------------------------------------------
# D) CONFIRMAR GIRO BAJISTA
# ------------------------------------------------

sig["RSI_Bajando"] = sig["RSI"] < sig["RSI"].shift(1)
sig["Precio_Bajando"] = sig["Close"] < sig["Close"].shift(1)

sig["Vuelve_Dentro_BB_Sup"] = (
    sig["Close"] < sig["BB_Sup"]
)

sig["Rompe_Min_Anterior"] = (
    sig["Close"] < sig["Low"].shift(1)
)

sig["Volumen_Salida_OK"] = (
    sig["Volume"] >= sig["Vol_Media"] * VOLUMEN_SALIDA_MULT
)

cond_salida = (
    sig["RSI_Bajando"] &
    sig["Precio_Bajando"] &
    sig["Vuelve_Dentro_BB_Sup"]
)

if EXIGIR_ROMPER_MINIMO_ANTERIOR:
    cond_salida &= sig["Rompe_Min_Anterior"]

if EXIGIR_VOLUMEN_SALIDA:
    cond_salida &= sig["Volumen_Salida_OK"]

sig["Giro_Bajista_Confirmado"] = cond_salida


print("✅ Señales del proceso preparadas")


In [ ]:

# ================================================================
# 🧪 CELDA 6 — MOTOR DE BACKTEST CON MÁQUINA DE ESTADOS
# ================================================================
#
# ESTADOS:
#
# FUERA
#   ↓
# VIGILANDO_SUELO
#   ↓
# EN_POSICION
#   ↓
# VIGILANDO_TECHO
#   ↓
# FUERA
#
# La señal se detecta al cierre.
# La compra/venta por señal se ejecuta en la apertura siguiente.
# El stop sí puede ejecutarse intradía usando Low.
# ================================================================

fecha_ini = pd.Timestamp(FECHA_OPERATIVA_INICIO)
fecha_fin = pd.Timestamp(FECHA_OPERATIVA_FIN)

if fecha_ini >= fecha_fin:
    raise ValueError("La fecha inicial debe ser anterior a la fecha final.")

capital = float(CAPITAL_INICIAL)

estado = "FUERA"

dias_vigilando_suelo = 0
dias_vigilando_techo = 0

minimo_suelo = None
entrada = {}

operaciones = []
eventos = []

ultima_salida_indice = -999


for i in range(1, len(sig) - 1):

    ayer = sig.iloc[i - 1]
    hoy = sig.iloc[i]
    manana = sig.iloc[i + 1]

    fecha_hoy = hoy["Date"]
    fecha_manana = manana["Date"]


    # ============================================================
    # 1) ESTADO FUERA
    # ============================================================

    if estado == "FUERA":

        # No buscamos entradas demasiado pronto tras una salida
        if i - ultima_salida_indice <= DIAS_ESPERA_TRAS_SALIDA:
            continue

        if bool(hoy["Caida_Fuerte"]):

            estado = "VIGILANDO_SUELO"
            dias_vigilando_suelo = 0
            minimo_suelo = hoy["Low"]

            eventos.append({
                "Fecha": fecha_hoy,
                "Evento": "CAÍDA FUERTE / VIGILAR SUELO",
                "Precio": hoy["Close"]
            })

        continue


    # ============================================================
    # 2) VIGILANDO SUELO
    # ============================================================

    if estado == "VIGILANDO_SUELO":

        dias_vigilando_suelo += 1

        # Actualizamos el mínimo de la caída
        minimo_suelo = min(minimo_suelo, hoy["Low"])

        # Si pasan demasiados días sin rebote, abandonamos vigilancia
        if dias_vigilando_suelo > DIAS_VIGILANCIA_SUELO:

            eventos.append({
                "Fecha": fecha_hoy,
                "Evento": "FIN VIGILANCIA SUELO SIN ENTRADA",
                "Precio": hoy["Close"]
            })

            estado = "FUERA"
            minimo_suelo = None
            continue

        # Confirmación de rebote.
        # La COMPRA se ejecuta mañana si mañana está dentro de ventana.
        if (
            bool(hoy["Rebote_Confirmado"]) and
            fecha_manana >= fecha_ini and
            fecha_manana <= fecha_fin
        ):

            precio_entrada = manana["Open"] * (1 + SLIPPAGE_PCT / 100)

            capital_para_operar = (
                capital if REINVERTIR_TODO else CAPITAL_INICIAL
            )

            factor_comision = 1 + COMISION_PCT / 100

            acciones = capital_para_operar / (
                precio_entrada * factor_comision
            )

            importe_compra = acciones * precio_entrada
            coste_compra = importe_compra * (COMISION_PCT / 100)

            capital_invertido_neto = importe_compra + coste_compra

            stop_inicial = minimo_suelo * (
                1 - STOP_MARGEN_PCT / 100
            )

            entrada = {
                "fecha": fecha_manana,
                "precio": precio_entrada,
                "acciones": acciones,
                "capital_antes": capital,
                "capital_invertido_neto": capital_invertido_neto,
                "stop_inicial": stop_inicial,
                "stop_actual": stop_inicial,
                "minimo_suelo": minimo_suelo
            }

            eventos.append({
                "Fecha": fecha_manana,
                "Evento": "COMPRA",
                "Precio": precio_entrada
            })

            estado = "EN_POSICION"
            dias_vigilando_suelo = 0
            minimo_suelo = None

        continue


    # ============================================================
    # 3) EN POSICIÓN
    # ============================================================

    if estado == "EN_POSICION":

        # --------------------------------------------------------
        # TRAILING ESTRUCTURAL
        # --------------------------------------------------------

        if USAR_TRAILING_ESTRUCTURAL and i >= TRAILING_MIN_SESIONES:

            ventana_min = sig.iloc[
                i - TRAILING_MIN_SESIONES : i
            ]["Low"].min()

            nuevo_stop = ventana_min * (
                1 - TRAILING_MARGEN_PCT / 100
            )

            # El stop nunca baja
            entrada["stop_actual"] = max(
                entrada["stop_actual"],
                nuevo_stop
            )

        # --------------------------------------------------------
        # STOP INTRADÍA
        # --------------------------------------------------------

        if hoy["Low"] <= entrada["stop_actual"]:

            precio_salida = entrada["stop_actual"] * (
                1 - SLIPPAGE_PCT / 100
            )

            importe_salida = entrada["acciones"] * precio_salida
            coste_salida = importe_salida * (COMISION_PCT / 100)

            capital_final_op = importe_salida - coste_salida

            ganancia_pct = (
                capital_final_op /
                entrada["capital_invertido_neto"] - 1
            ) * 100

            capital = capital_final_op

            operaciones.append({
                "Entrada": entrada["fecha"],
                "Precio entrada": entrada["precio"],
                "Salida": fecha_hoy,
                "Precio salida": precio_salida,
                "Motivo salida": "STOP",
                "Ganancia %": ganancia_pct,
                "Capital antes": entrada["capital_antes"],
                "Capital después": capital,
                "Stop inicial": entrada["stop_inicial"],
                "Stop final": entrada["stop_actual"],
                "Mínimo suelo": entrada["minimo_suelo"]
            })

            eventos.append({
                "Fecha": fecha_hoy,
                "Evento": "SALIDA STOP",
                "Precio": precio_salida
            })

            estado = "FUERA"
            entrada = {}
            ultima_salida_indice = i
            continue

        # --------------------------------------------------------
        # POSIBLE TECHO
        # --------------------------------------------------------

        if bool(hoy["Posible_Techo"]):

            estado = "VIGILANDO_TECHO"
            dias_vigilando_techo = 0

            eventos.append({
                "Fecha": fecha_hoy,
                "Evento": "POSIBLE TECHO",
                "Precio": hoy["Close"]
            })

        continue


    # ============================================================
    # 4) VIGILANDO TECHO
    # ============================================================

    if estado == "VIGILANDO_TECHO":

        dias_vigilando_techo += 1

        # Seguimos protegiendo la posición con trailing
        if USAR_TRAILING_ESTRUCTURAL and i >= TRAILING_MIN_SESIONES:

            ventana_min = sig.iloc[
                i - TRAILING_MIN_SESIONES : i
            ]["Low"].min()

            nuevo_stop = ventana_min * (
                1 - TRAILING_MARGEN_PCT / 100
            )

            entrada["stop_actual"] = max(
                entrada["stop_actual"],
                nuevo_stop
            )

        # STOP
        if hoy["Low"] <= entrada["stop_actual"]:

            precio_salida = entrada["stop_actual"] * (
                1 - SLIPPAGE_PCT / 100
            )

            importe_salida = entrada["acciones"] * precio_salida
            coste_salida = importe_salida * (COMISION_PCT / 100)

            capital_final_op = importe_salida - coste_salida

            ganancia_pct = (
                capital_final_op /
                entrada["capital_invertido_neto"] - 1
            ) * 100

            capital = capital_final_op

            operaciones.append({
                "Entrada": entrada["fecha"],
                "Precio entrada": entrada["precio"],
                "Salida": fecha_hoy,
                "Precio salida": precio_salida,
                "Motivo salida": "STOP",
                "Ganancia %": ganancia_pct,
                "Capital antes": entrada["capital_antes"],
                "Capital después": capital,
                "Stop inicial": entrada["stop_inicial"],
                "Stop final": entrada["stop_actual"],
                "Mínimo suelo": entrada["minimo_suelo"]
            })

            eventos.append({
                "Fecha": fecha_hoy,
                "Evento": "SALIDA STOP",
                "Precio": precio_salida
            })

            estado = "FUERA"
            entrada = {}
            ultima_salida_indice = i
            continue

        # Confirmación de giro bajista:
        # señal hoy -> venta mañana.
        if (
            bool(hoy["Giro_Bajista_Confirmado"]) and
            fecha_manana <= fecha_fin
        ):

            precio_salida = manana["Open"] * (
                1 - SLIPPAGE_PCT / 100
            )

            importe_salida = entrada["acciones"] * precio_salida
            coste_salida = importe_salida * (COMISION_PCT / 100)

            capital_final_op = importe_salida - coste_salida

            ganancia_pct = (
                capital_final_op /
                entrada["capital_invertido_neto"] - 1
            ) * 100

            capital = capital_final_op

            operaciones.append({
                "Entrada": entrada["fecha"],
                "Precio entrada": entrada["precio"],
                "Salida": fecha_manana,
                "Precio salida": precio_salida,
                "Motivo salida": "GIRO BAJISTA",
                "Ganancia %": ganancia_pct,
                "Capital antes": entrada["capital_antes"],
                "Capital después": capital,
                "Stop inicial": entrada["stop_inicial"],
                "Stop final": entrada["stop_actual"],
                "Mínimo suelo": entrada["minimo_suelo"]
            })

            eventos.append({
                "Fecha": fecha_manana,
                "Evento": "VENTA GIRO BAJISTA",
                "Precio": precio_salida
            })

            estado = "FUERA"
            entrada = {}
            ultima_salida_indice = i
            continue

        # Si no confirma techo en X días,
        # volvemos a estado EN_POSICION.
        if dias_vigilando_techo > DIAS_VIGILANCIA_TECHO:

            eventos.append({
                "Fecha": fecha_hoy,
                "Evento": "TECHO NO CONFIRMADO / SEGUIR",
                "Precio": hoy["Close"]
            })

            estado = "EN_POSICION"
            dias_vigilando_techo = 0

        continue


# ================================================================
# REGLA FINAL DE VENTANA
# ================================================================

operacion_anulada = None

if estado in ["EN_POSICION", "VIGILANDO_TECHO"] and entrada:

    operacion_anulada = {
        "Entrada anulada": entrada["fecha"],
        "Precio entrada": entrada["precio"],
        "Motivo": "Seguía abierta al finalizar la ventana operativa"
    }

    # Restauramos capital previo.
    capital = entrada["capital_antes"]

    eventos.append({
        "Fecha": fecha_fin,
        "Evento": "OPERACIÓN ANULADA FIN VENTANA",
        "Precio": np.nan
    })

    estado = "FUERA"
    entrada = {}


ops = pd.DataFrame(operaciones)
eventos_df = pd.DataFrame(eventos)

print("=" * 80)
print("✅ BACKTEST TERMINADO")
print("=" * 80)
print(f"Capital inicial: ${CAPITAL_INICIAL:,.2f}")
print(f"Capital final:   ${capital:,.2f}")
print(f"Rentabilidad:    {(capital/CAPITAL_INICIAL - 1)*100:+.2f}%")
print(f"Operaciones:     {len(ops)}")

if operacion_anulada:
    print()
    print("⚠️ OPERACIÓN FINAL ANULADA")
    print(f"Entrada: {operacion_anulada['Entrada anulada'].date()}")
    print(f"Precio:  ${operacion_anulada['Precio entrada']:.2f}")


In [ ]:

# ================================================================
# 📋 CELDA 7 — TABLA DE OPERACIONES
# ================================================================

if ops.empty:

    print("No se completó ninguna operación dentro de la ventana.")

else:

    tabla = ops.copy()

    for col in [
        "Precio entrada",
        "Precio salida",
        "Stop inicial",
        "Stop final",
        "Mínimo suelo"
    ]:
        tabla[col] = tabla[col].map(lambda x: f"${x:,.2f}")

    tabla["Ganancia %"] = tabla["Ganancia %"].map(
        lambda x: f"{x:+.2f}%"
    )

    tabla["Capital antes"] = tabla["Capital antes"].map(
        lambda x: f"${x:,.2f}"
    )

    tabla["Capital después"] = tabla["Capital después"].map(
        lambda x: f"${x:,.2f}"
    )

    display(tabla)

if operacion_anulada:

    print()
    print("Entrada anulada por quedar abierta al final de la ventana:")
    display(pd.DataFrame([operacion_anulada]))


In [ ]:

# ================================================================
# 🧭 CELDA 8 — EVENTOS DEL ALGORITMO
# ================================================================
# Muy útil para entender POR QUÉ entró o no entró.

if eventos_df.empty:
    print("No se detectaron eventos.")

else:
    display(eventos_df)


In [ ]:

# ================================================================
# 📊 CELDA 9 — RESUMEN ESTADÍSTICO
# ================================================================

if ops.empty:

    print("Sin operaciones cerradas para calcular estadísticas.")

else:

    positivas = (ops["Ganancia %"] > 0).sum()
    negativas = (ops["Ganancia %"] <= 0).sum()

    acierto = positivas / len(ops) * 100

    mejor = ops["Ganancia %"].max()
    peor = ops["Ganancia %"].min()
    media = ops["Ganancia %"].mean()

    equity = pd.Series(
        [CAPITAL_INICIAL] + ops["Capital después"].tolist(),
        dtype=float
    )

    max_acum = equity.cummax()
    drawdown = (equity / max_acum - 1) * 100
    max_drawdown = drawdown.min()

    print("=" * 65)
    print(f"📊 RESULTADOS — {TICKER}")
    print("=" * 65)
    print(f"Capital inicial:       ${CAPITAL_INICIAL:,.2f}")
    print(f"Capital final:         ${capital:,.2f}")
    print(f"Rentabilidad total:    {(capital/CAPITAL_INICIAL - 1)*100:+.2f}%")
    print(f"Operaciones cerradas:  {len(ops)}")
    print(f"Ganadoras:             {positivas}")
    print(f"Perdedoras:            {negativas}")
    print(f"Acierto:               {acierto:.1f}%")
    print(f"Ganancia media/op.:    {media:+.2f}%")
    print(f"Mejor operación:       {mejor:+.2f}%")
    print(f"Peor operación:        {peor:+.2f}%")
    print(f"Drawdown aprox.:       {max_drawdown:.2f}%")
    print("=" * 65)


In [ ]:

# ================================================================
# 📈 CELDA 10 — GRÁFICA PRINCIPAL
# ================================================================

plt.figure(figsize=(17, 9))

plt.plot(
    sig["Date"],
    sig["Close"],
    linewidth=1.4,
    label=f"{TICKER} Close"
)

# Bollinger
plt.plot(
    sig["Date"],
    sig["BB_Inf"],
    linewidth=0.8,
    alpha=0.55,
    label="Bollinger inferior"
)

plt.plot(
    sig["Date"],
    sig["BB_Sup"],
    linewidth=0.8,
    alpha=0.55,
    label="Bollinger superior"
)

# Ventana operativa
plt.axvspan(
    fecha_ini,
    fecha_fin,
    alpha=0.08,
    label="Ventana operativa"
)

# Entradas/salidas
if not ops.empty:

    plt.scatter(
        ops["Entrada"],
        ops["Precio entrada"],
        marker="^",
        s=100,
        label="Compra",
        zorder=6
    )

    plt.scatter(
        ops["Salida"],
        ops["Precio salida"],
        marker="v",
        s=100,
        label="Venta",
        zorder=6
    )

# Caídas fuertes detectadas
caidas = sig[sig["Caida_Fuerte"]]

if not caidas.empty:
    plt.scatter(
        caidas["Date"],
        caidas["Close"],
        marker="o",
        s=35,
        alpha=0.6,
        label="Caída fuerte"
    )

# Posibles techos
techos = sig[sig["Posible_Techo"]]

if not techos.empty:
    plt.scatter(
        techos["Date"],
        techos["Close"],
        marker="s",
        s=35,
        alpha=0.6,
        label="Posible techo"
    )

plt.title(
    f"{TICKER} — Rebote RSI + Bollinger + Volumen | "
    f"{FECHA_OPERATIVA_INICIO} → {FECHA_OPERATIVA_FIN}"
)

plt.xlabel("Fecha")
plt.ylabel("Precio ($)")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# 📉 CELDA 11 — RSI
# ================================================================

plt.figure(figsize=(16, 5))

plt.plot(
    sig["Date"],
    sig["RSI"],
    linewidth=1.2
)

plt.axhline(RSI_SUELO, linestyle="--", alpha=0.5)
plt.axhline(RSI_TECHO, linestyle="--", alpha=0.5)

plt.axvspan(
    fecha_ini,
    fecha_fin,
    alpha=0.08
)

plt.title(f"{TICKER} — RSI")
plt.xlabel("Fecha")
plt.ylabel("RSI")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# 📊 CELDA 12 — VOLUMEN
# ================================================================

plt.figure(figsize=(16, 5))

plt.bar(
    sig["Date"],
    sig["Volume"],
    alpha=0.45,
    label="Volumen"
)

plt.plot(
    sig["Date"],
    sig["Vol_Media"],
    linewidth=1.2,
    label=f"Media volumen {VOLUMEN_PERIODO}"
)

plt.axvspan(
    fecha_ini,
    fecha_fin,
    alpha=0.08
)

plt.title(f"{TICKER} — Volumen")
plt.xlabel("Fecha")
plt.ylabel("Volumen")
plt.grid(True, alpha=0.20)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# 💰 CELDA 13 — CURVA DE CAPITAL
# ================================================================

if ops.empty:

    print("No hay operaciones cerradas para dibujar la curva de capital.")

else:

    fechas_equity = [
        pd.Timestamp(FECHA_OPERATIVA_INICIO)
    ] + ops["Salida"].tolist()

    valores_equity = [
        CAPITAL_INICIAL
    ] + ops["Capital después"].tolist()

    plt.figure(figsize=(14, 6))

    plt.plot(
        fechas_equity,
        valores_equity,
        marker="o"
    )

    plt.title(f"{TICKER} — Evolución del capital")
    plt.xlabel("Fecha")
    plt.ylabel("Capital ($)")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:

# ================================================================
# 💾 CELDA 14 — GUARDAR RESULTADOS
# ================================================================

if ops.empty:

    print("No hay operaciones cerradas para guardar.")

else:

    nombre = (
        f"REBOTES_{TICKER}_"
        f"{FECHA_OPERATIVA_INICIO}_"
        f"{FECHA_OPERATIVA_FIN}.csv"
    )

    ops.to_csv(nombre, index=False)

    print(f"✅ Resultados guardados en: {nombre}")



## 🧠 Qué hace exactamente esta estrategia

### Entrada

Primero busca una caída fuerte:

- RSI bajo
- cierre en/bajo Bollinger inferior
- caída acumulada relevante

Eso **NO compra**.

Activa un estado de vigilancia durante varios días.

Solo compra cuando aparece un rebote confirmado:

- RSI subiendo
- precio subiendo
- precio vuelve dentro de Bollinger
- opcionalmente rompe el máximo de la vela anterior
- volumen superior a su media

La orden se ejecuta en la apertura de la sesión siguiente.

### Salida

Primero busca una posible zona de techo:

- RSI elevado
- cierre en/sobre Bollinger superior
- subida acumulada reciente

Eso **NO vende todavía**.

Después espera un giro bajista:

- RSI bajando
- precio bajando
- vuelve dentro desde Bollinger superior
- opcionalmente rompe el mínimo de la vela anterior
- volumen relevante

La venta se ejecuta en la apertura siguiente.

### Stop

El stop inicial se coloca debajo del mínimo observado durante la fase de suelo.  
También puede ir subiendo bajo los mínimos recientes mediante `USAR_TRAILING_ESTRUCTURAL`.

### Importante

Esta estrategia busca **confirmación**, no acertar exactamente el mínimo o máximo. Por diseño, normalmente comprará por encima del mínimo real y venderá por debajo del máximo real. La idea es reducir entradas prematuras durante caídas violentas.
